In [31]:
import os
import re
import time
import json
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm
from openai import OpenAI
import matplotlib.pyplot as plt

In [ ]:
# API setup
# Put your NVIDIA API key here or set it as an environment variable

os.environ["NVIDIA_API_KEY"] = ""

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"],
)

MODEL_NAME = "openai/gpt-oss-20b"

In [33]:
def get_completion(prompt, temperature=0, max_tokens=800):
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )

        content = response.choices[0].message.content

        # Fallback for reasoning models
        if not content or content.strip() == "":
            if hasattr(response.choices[0].message, "reasoning_content"):
                content = response.choices[0].message.reasoning_content

        return content

    except Exception as e:
        print("Error:", e)
        return None

In [34]:
test_prompt = """
You are a biomedical reasoning assistant.
Briefly explain what a differentially expressed gene means.
"""

response = get_completion(test_prompt)
print(response)

A **differentially expressed gene** is one whose level of transcription (mRNA abundance) differs significantly between two or more biological conditions—such as healthy vs. diseased tissue, treated vs. untreated cells, or different developmental stages. In practice, researchers measure gene expression (e.g., via RNA‑seq or microarrays) and use statistical tests to identify genes whose expression changes exceed a chosen threshold of fold‑change and statistical significance (p‑value or false discovery rate). These genes are considered to be “differentially expressed” because their activity is altered in the context being studied.


In [35]:
def run_direct_gene_annotation(up_genes, down_genes):
    start_time = time.time()

    prompt = f"""
You are a biomedical genomics expert.

You will receive two gene lists from differential expression analysis comparing
normal colorectal mucosa and colorectal adenoma.

Upregulated genes in adenoma:
{", ".join(up_genes)}

Downregulated genes in adenoma:
{", ".join(down_genes)}

Task:
Identify the main biological pathways or processes represented by these genes.

Return ONLY a markdown table.
Do not write explanations before or after the table.
Do not write your reasoning process.

The table must have exactly these columns:
Pathway_or_Process | Direction | Supporting_Genes | Explanation | Confidence

Rules:
- Use only genes from the provided lists.
- Do not invent genes.
- Do not claim causality.
- Do not claim statistical significance for pathways.
- Keep each explanation one sentence only.
- Confidence must be High, Medium, or Low.
"""

    raw = get_completion(prompt, temperature=0, max_tokens=1800)
    duration = time.time() - start_time

    return raw, duration

In [36]:
def run_reasoning_gene_annotation(up_genes, down_genes):
    start_time = time.time()

    prompt = f"""
You are a biomedical genomics expert.

Given the following differentially expressed genes from colorectal adenoma:

Upregulated genes:
{", ".join(up_genes)}

Downregulated genes:
{", ".join(down_genes)}

Task:
Generate a structured biomedical interpretation of the gene lists.

STRICT OUTPUT FORMAT:
Return only a markdown table.
The first line must be:
| Pathway_or_Process | Direction | Supporting_Genes | Biological_Rationale | Confidence |

Do not write any text before the table.
Do not write any text after the table.
Do not explain your reasoning process.
Do not include phrases such as "We need to", "Let's propose", or "Potential processes".

Rules:
- Use only genes provided in the input lists.
- Do not add genes that were not provided.
- Do not claim statistical significance for pathways.
- Avoid broad claims unless supported by multiple genes.
- Confidence must be High, Medium, or Low.
- Return 5 to 8 rows only.
"""

    raw = get_completion(prompt, temperature=0, max_tokens=1800)
    duration = time.time() - start_time

    return raw, duration

In [37]:
def run_self_consistency_gene_annotation(up_genes, down_genes, k=3):
    start_time = time.time()
    outputs = []

    for i in range(k):
        prompt = f"""
Return exactly 5 lines only.

Format:
1. Pathway_or_Process: Gene1, Gene2, Gene3
2. Pathway_or_Process: Gene1, Gene2, Gene3
3. Pathway_or_Process: Gene1, Gene2, Gene3
4. Pathway_or_Process: Gene1, Gene2, Gene3
5. Pathway_or_Process: Gene1, Gene2, Gene3

Gene lists:

Upregulated genes in adenoma:
{", ".join(up_genes)}

Downregulated genes in adenoma:
{", ".join(down_genes)}

Rules:
- Do not explain.
- Do not write reasoning.
- Do not write introduction.
- Use only genes from the provided lists.
- Return only the 5 numbered lines.
"""

        raw = get_completion(prompt, temperature=0.2, max_tokens=400)
        outputs.append(raw)

    duration = time.time() - start_time
    return outputs, duration

In [38]:
def run_gene_annotation_verification(up_genes, down_genes, annotation_output):
    start_time = time.time()

    prompt = f"""
You are a strict biomedical verification assistant.

Input gene lists:

Upregulated genes:
{", ".join(up_genes)}

Downregulated genes:
{", ".join(down_genes)}

Proposed LLM annotation:
{annotation_output}

Task:
Verify each pathway/process in the proposed annotation.

Return ONLY a markdown table.
Do not write any text before or after the table.

The table must have exactly these columns:
Pathway_or_Process | Verification_Status | Problem_Found | Corrected_Comment

Verification_Status must be one of:
Supported, Partially Supported, Unsupported

Rules:
- Check whether the supporting genes are present in the input lists.
- Check whether the gene-pathway relationship is biologically plausible.
- Mark broad or weakly supported claims as Partially Supported.
- Mark unsupported or invented claims as Unsupported.
- Do not add new pathways.
"""

    raw = get_completion(prompt, temperature=0, max_tokens=1800)
    duration = time.time() - start_time

    return raw, duration

# Phase 2: Load GEO2R Results


In [39]:
# Phase 2: Load GEO2R Results

geo_file = "/content/GSE8671.top.table.tsv"

df = pd.read_csv(geo_file, sep="\t")

print("Shape:", df.shape)
df.head()

Shape: (54675, 8)


,ID,adj.P.Val,P.Value,t,B,logFC,Gene.symbol,Gene.title
0,222696_at,9.880000e-35,1.810000e-39,-29.998043,79.021858,-2.589510,AXIN2,axin 2
1,203256_at,2.810000e-31,1.030000e-35,-25.930088,70.735590,-6.372120,CDH3,cadherin 3
2,212014_x_at,8.610000e-31,5.560000e-35,-25.193106,69.105643,-2.142557,CD44,CD44 molecule (Indian blood group)
3,209835_x_at,8.610000e-31,6.300000e-35,-25.139073,68.984450,-2.059452,CD44,CD44 molecule (Indian blood group)
4,212942_s_at,1.030000e-30,9.400000e-35,-24.967117,68.597200,-6.013322,CEMIP,cell migration inducing hyaluronan binding pro...


In [40]:
df.columns

Index(['ID', 'adj.P.Val', 'P.Value', 't', 'B', 'logFC', 'Gene.symbol',
       'Gene.title'],
      dtype='object')

In [41]:
# Keep required columns only
df = df[["ID", "adj.P.Val", "P.Value", "t", "B", "logFC", "Gene.symbol", "Gene.title"]].copy()

# Remove rows without gene symbol
df = df.dropna(subset=["Gene.symbol", "logFC", "adj.P.Val"])

# Some rows may contain multiple gene symbols separated by ///
# We take the first gene symbol for simplicity
df["Gene.symbol"] = df["Gene.symbol"].astype(str).str.split("///").str[0].str.strip()

# Remove empty symbols
df = df[df["Gene.symbol"] != ""]

df.head()

,ID,adj.P.Val,P.Value,t,B,logFC,Gene.symbol,Gene.title
0,222696_at,9.880000e-35,1.810000e-39,-29.998043,79.021858,-2.589510,AXIN2,axin 2
1,203256_at,2.810000e-31,1.030000e-35,-25.930088,70.735590,-6.372120,CDH3,cadherin 3
2,212014_x_at,8.610000e-31,5.560000e-35,-25.193106,69.105643,-2.142557,CD44,CD44 molecule (Indian blood group)
3,209835_x_at,8.610000e-31,6.300000e-35,-25.139073,68.984450,-2.059452,CD44,CD44 molecule (Indian blood group)
4,212942_s_at,1.030000e-30,9.400000e-35,-24.967117,68.597200,-6.013322,CEMIP,cell migration inducing hyaluronan binding pro...


In [42]:
# DEG filtering criteria
adj_p_threshold = 0.05
logfc_threshold = 1.5

deg = df[
    (df["adj.P.Val"] < adj_p_threshold) &
    (df["logFC"].abs() > logfc_threshold)
].copy()

# Based on GEO2R output direction:
# Negative logFC appears to represent genes upregulated in adenoma
adenoma_up = deg[deg["logFC"] < -logfc_threshold].copy()
adenoma_down = deg[deg["logFC"] > logfc_threshold].copy()

print("Total significant DEGs:", deg.shape[0])
print("Upregulated in adenoma:", adenoma_up.shape[0])
print("Downregulated in adenoma:", adenoma_down.shape[0])

Total significant DEGs: 1987
Upregulated in adenoma: 466
Downregulated in adenoma: 1521


In [43]:
adenoma_up_unique = (
    adenoma_up
    .sort_values("adj.P.Val")
    .drop_duplicates(subset="Gene.symbol", keep="first")
)

adenoma_down_unique = (
    adenoma_down
    .sort_values("adj.P.Val")
    .drop_duplicates(subset="Gene.symbol", keep="first")
)

print("Unique upregulated genes in adenoma:", adenoma_up_unique.shape[0])
print("Unique downregulated genes in adenoma:", adenoma_down_unique.shape[0])

Unique upregulated genes in adenoma: 365
Unique downregulated genes in adenoma: 1054


In [44]:
top_up_df = adenoma_up_unique.sort_values("adj.P.Val").head(50)
top_down_df = adenoma_down_unique.sort_values("adj.P.Val").head(50)

top_up_genes = top_up_df["Gene.symbol"].tolist()
top_down_genes = top_down_df["Gene.symbol"].tolist()

print("Top upregulated genes in adenoma:")
print(top_up_genes)

print("\nTop downregulated genes in adenoma:")
print(top_down_genes)

Top upregulated genes in adenoma:
['AXIN2', 'CDH3', 'CD44', 'CEMIP', 'SORD', 'GTF2IRD1', 'GALNT6', 'FOXQ1', 'ANXA3', 'RNF43', 'TCN1', 'ENC1', 'MMP7', 'MYC', 'GPSM2', 'MET', 'TRIM16', 'SLC12A2', 'NFE2L3', 'TBX3', 'S100A11', 'LOC101928195', 'CLDN1', 'ECT2', 'C2CD4A', 'BACE2', 'EPB41L2', 'TEAD4', 'AJUBA', 'QPCT', 'TACSTD2', 'CENPW', 'ANKRD22', 'S100A2', 'TGFBI', 'ASCL2', 'TKT', 'RPP40', 'SNORD88C', 'CADPS', 'ITGA6', 'NEBL', 'NQO1', 'RRM2', 'PHF19', 'CDK1', 'PRC1', 'ANLN', 'CEP55', 'TDGF1P3']

Top downregulated genes in adenoma:
['ADAMDEC1', 'LPAR1', 'PAG1', 'ABCG2', 'PRDX6', 'PHLPP2', 'ABCA8', 'PCK1', 'EDIL3', 'APPL2', 'TSPAN7', 'MYLK', 'CNNM2', 'CLDN23', 'C2orf88', 'PYY', 'RELL1', 'TMCC3', 'TNS1', 'SYNE3', 'SLC4A4', 'SLC17A4', 'MGAT4A', 'EDN3', 'HAPLN1', 'CPNE8', 'SECTM1', 'TPH1', 'IL6R', 'TP53INP2', 'MEIS1', 'AQP8', 'SORBS2', 'TRPM6', 'TMEM171', 'DHRS11', 'DPT', 'GUCA2B', 'GPX3', 'MAMDC2', 'GCNT2', 'ANK2', 'CA7', 'NR3C1', 'CLIC5', 'GUCA2A', 'PLP1', 'CDKN2B', 'SCGN', 'HIGD1A']


In [45]:
deg.to_csv("GSE8671_significant_DEGs.csv", index=False)
adenoma_up_unique.to_csv("GSE8671_adenoma_upregulated_genes.csv", index=False)
adenoma_down_unique.to_csv("GSE8671_adenoma_downregulated_genes.csv", index=False)

top_up_df.to_csv("GSE8671_top50_adenoma_upregulated.csv", index=False)
top_down_df.to_csv("GSE8671_top50_adenoma_downregulated.csv", index=False)

In [46]:
reasoning_output, reasoning_time = run_reasoning_gene_annotation(top_up_genes, top_down_genes)

print("Running time:", round(reasoning_time, 2), "seconds")
print(reasoning_output)

Running time: 6.54 seconds
| Pathway_or_Process | Direction | Supporting_Genes | Biological_Rationale | Confidence |
|---------------------|-----------|-------------------|----------------------|------------|
| Wnt signaling activation | Up | AXIN2, MYC, RNF43, CD44, CDH3, TACSTD2 | Upregulation of canonical Wnt target genes and stem‑cell markers indicates enhanced Wnt pathway activity driving proliferation and stemness in adenoma cells. | High |
| Cell cycle/proliferation | Up | CDK1, PRC1, ANLN, CEP55, RRM2, MYC | Elevated expression of mitotic regulators and DNA synthesis enzymes reflects increased cell division and tumor growth. | High |
| EMT / invasion | Up | MMP7, S100A2, S100A11, CLDN1, ITGA6, TACSTD2 | Upregulation of matrix metalloproteinase, adhesion molecules, and epithelial markers suggests acquisition of EMT‑like traits and invasive potential. | Medium |
| Loss of secretory/goblet cell differentiation | Down | PYY, TPH1, GUCA2A, GUCA2B, AQP8, SLC4A4, SLC17A4, CLDN23 | Dow

In [47]:
sc_outputs, sc_time = run_self_consistency_gene_annotation(top_up_genes, top_down_genes, k=3)

print("Running time:", round(sc_time, 2), "seconds")

for i, output in enumerate(sc_outputs, 1):
    print(f"\n--- Run {i} ---")
    print(output)

Running time: 4.18 seconds

--- Run 1 ---
1. Wnt signaling: AXIN2, MYC, CD44  
2. Cell adhesion: CDH3, CLDN1, ITGA6  
3. ECM remodeling: MMP7, S100A11, TGFBI  
4. Proliferation: CDK1, PRC1, RRM2  
5. Stemness: ASCL2, FOXQ1, ANXA3

--- Run 2 ---
1. Wnt signaling: AXIN2, RNF43, MYC  
2. Cell adhesion: CDH3, CLDN1, ITGA6  
3. Proliferation: CDK1, PRC1, ANLN  
4. ECM remodeling: MMP7, TGFBI, S100A11  
5. Metabolism: TKT, NQO1, RRM2

--- Run 3 ---
We need to produce exactly 5 lines, each line numbered 1-5, format: "1. Pathway_or_Process: Gene1, Gene2, Gene3". We must use only genes from the provided lists. We need to choose pathways or processes? The prompt: "Return exactly 5 lines only. Format: 1. Pathway_or_Process: Gene1, Gene2, Gene3". So we need to list 5 pathways or processes, each with 3 genes. Genes must be from the lists. We can choose any pathway names? They didn't specify which pathways. We can just create plausible ones like "Wnt signaling", "Cell adhesion", etc. Genes must be f

In [48]:
verification_output, verification_time = run_gene_annotation_verification(
    top_up_genes,
    top_down_genes,
    reasoning_output
)

print("Running time:", round(verification_time, 2), "seconds")
print(verification_output)

Running time: 3.89 seconds
| Pathway_or_Process | Verification_Status | Problem_Found | Corrected_Comment |
|---------------------|---------------------|---------------|-------------------|
| Wnt signaling activation | Supported | None | No issues |
| Cell cycle/proliferation | Supported | None | No issues |
| EMT / invasion | Supported | None | No issues |
| Loss of secretory/goblet cell differentiation | Supported | None | No issues |
| Downregulation of metabolic/antioxidant genes | Supported | None | No issues |


In [49]:
results = {
    "deg_summary": {
        "total_significant_DEGs": int(deg.shape[0]),
        "upregulated_in_adenoma": int(adenoma_up.shape[0]),
        "downregulated_in_adenoma": int(adenoma_down.shape[0]),
        "unique_upregulated_in_adenoma": int(adenoma_up_unique.shape[0]),
        "unique_downregulated_in_adenoma": int(adenoma_down_unique.shape[0])
    },
    "top_up_genes": top_up_genes,
    "top_down_genes": top_down_genes,
    "reasoning_output": reasoning_output,
    "reasoning_time": reasoning_time,
    "self_consistency_outputs": sc_outputs,
    "self_consistency_time": sc_time,
    "verification_output": verification_output,
    "verification_time": verification_time
}

with open("llm_gene_reasoning_outputs_final.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

In [50]:
with open("llm_gene_reasoning_outputs_final.json", "r", encoding="utf-8") as f:
    saved_results = json.load(f)

print(saved_results["reasoning_output"])

| Pathway_or_Process | Direction | Supporting_Genes | Biological_Rationale | Confidence |
|---------------------|-----------|-------------------|----------------------|------------|
| Wnt signaling activation | Up | AXIN2, MYC, RNF43, CD44, CDH3, TACSTD2 | Upregulation of canonical Wnt target genes and stem‑cell markers indicates enhanced Wnt pathway activity driving proliferation and stemness in adenoma cells. | High |
| Cell cycle/proliferation | Up | CDK1, PRC1, ANLN, CEP55, RRM2, MYC | Elevated expression of mitotic regulators and DNA synthesis enzymes reflects increased cell division and tumor growth. | High |
| EMT / invasion | Up | MMP7, S100A2, S100A11, CLDN1, ITGA6, TACSTD2 | Upregulation of matrix metalloproteinase, adhesion molecules, and epithelial markers suggests acquisition of EMT‑like traits and invasive potential. | Medium |
| Loss of secretory/goblet cell differentiation | Down | PYY, TPH1, GUCA2A, GUCA2B, AQP8, SLC4A4, SLC17A4, CLDN23 | Downregulation of enteroendocr

In [51]:
print(saved_results["verification_output"])

| Pathway_or_Process | Verification_Status | Problem_Found | Corrected_Comment |
|---------------------|---------------------|---------------|-------------------|
| Wnt signaling activation | Supported | None | No issues |
| Cell cycle/proliferation | Supported | None | No issues |
| EMT / invasion | Supported | None | No issues |
| Loss of secretory/goblet cell differentiation | Supported | None | No issues |
| Downregulation of metabolic/antioxidant genes | Supported | None | No issues |
